# E9: Integrated Pipeline — Inspection, Evaluation & Error Analysis

## Three modes

1. **Inspection** (Section 2): Run ONE game step-by-step, print every decision
2. **Evaluation** (Section 3): Run N games, collect metrics
3. **Error Analysis** (Section 4): Post-hoc study of failures

## Pipeline under test

```
RETRIEVE:  causal_search → k episodes with continuations
VALIDATE:  KG.filter_valid_actions → remove invalid actions
SCORE:     BN.query → P(success | task, phase, region, action)
ACT:       LLM chooses from scored + validated actions
LEARN:     outcome → typed_edges + BN.observe + WeightLearner
```

**Model**: Ollama qwen2.5:7b (zero cost)


In [ ]:
import os, json, time, re
import numpy as np
from collections import Counter, defaultdict
from sentence_transformers import SentenceTransformer
from openai import OpenAI

# === CONFIG ===
OLLAMA_URL = 'http://hpc.glaciar.lab:11434/v1'
client = OpenAI(base_url=OLLAMA_URL, api_key='ollama')
LLM_MODEL = 'qwen2.5-coder:7b-instruct'

MAX_STEPS = 30
N_EVAL = 30
TOP_K = 3

embedder = SentenceTransformer('all-MiniLM-L6-v2')
print(f'LLM: {LLM_MODEL}')
print(f'Embedder: {embedder.get_sentence_embedding_dimension()}D')


## 1. Build Memory Components

Three components initialized from expert data:
- **CVX Index**: 336 expert episodes with temporal edges
- **Knowledge Graph**: task structure with preconditions/effects  
- **Bayesian Network**: P(success | task_type, phase, action)


In [ ]:
import chronos_vector as cvx

# === CVX Index ===
with open('../data/episodic/alfworld_metadata.json') as f:
    meta = json.load(f)

index = cvx.TemporalIndex(m=16, ef_construction=100)
action_lookup, task_lookup = {}, {}

for ep_id_str, ep in meta.items():
    ep_id = int(ep_id_str)
    task = ep.get('task', '')
    task_lookup[ep_id] = task
    for si, step in enumerate(ep['steps']):
        text = f"{task} | {step.get('action','')} | {step.get('observation','')}"
        vec = embedder.encode(text).tolist()
        ts = ep_id * 10000 + si
        index.insert(ep_id, ts, vec, reward=1.0)
        action_lookup[ts] = step.get('action', '')

print(f'CVX Index: {len(index)} points from {len(meta)} episodes')
print(f'Action lookup: {len(action_lookup)} entries')


In [ ]:
# === Task type detection ===
TASK_TYPES = {
    'heat': 'heat', 'cool': 'cool', 'clean': 'clean',
    'examine': 'examine', 'look at': 'examine',
    'find two': 'pick_two', 'put two': 'pick_two',
}

def detect_task_type(task_text):
    for kw, tt in TASK_TYPES.items():
        if kw in task_text.lower(): return tt
    return 'pick'

STRATEGIES = {
    'pick': 'Find the object, take it, go to target, put it.',
    'heat': 'Find object, take, go to microwave, heat, take, go to target, put.',
    'cool': 'Find object, take, go to fridge, cool, take, go to target, put.',
    'clean': 'Find object, take, go to sinkbasin, clean, go to target, put.',
    'examine': 'Find object, take, find lamp, use lamp.',
    'pick_two': 'Find first, take, put at target. Find second, take, put at target.',
}

# === Phase detection from observation ===
def detect_phase(observation, history):
    """Infer abstract phase from recent action history."""
    if not history:
        return 'searching'
    
    recent = [h['action'].lower() for h in history[-3:]]
    last_action = recent[-1] if recent else ''
    
    # Just transformed (heated/cooled/cleaned) → placing
    if any(a.startswith(('heat', 'cool', 'clean')) and not a.startswith('go') for a in recent):
        if any(a.startswith('take') for a in recent[recent.index(next(a for a in recent if a.startswith(('heat','cool','clean')))):]):
            return 'placing'
    
    # Holding something → check if we need to transform or place
    if any(a.startswith(('take', 'pick')) for a in recent[-2:]):
        return 'holding'
    
    # At an appliance → transforming
    obs_lower = observation.lower()
    if any(x in obs_lower for x in ['microwave', 'fridge', 'sinkbasin']):
        if any(a.startswith(('take', 'pick')) for a in recent[-4:]):
            return 'transforming'
    
    return 'searching'

# === Action type classification ===
def classify_action(action_text):
    a = action_text.lower()
    if a.startswith('go to'): return 'navigate'
    if a.startswith('take') or a.startswith('pick'): return 'take'
    if a.startswith('put'): return 'place'
    if a.startswith('open'): return 'open'
    if a.startswith('close'): return 'close'
    if a.startswith('clean'): return 'clean'
    if a.startswith('heat') or a.startswith('cook'): return 'heat'
    if a.startswith('cool'): return 'cool'
    if a.startswith('use'): return 'use'
    if a.startswith('examine') or a.startswith('look'): return 'examine'
    return 'other'

print('Phase detection + action classification ready')
print(f'Task types: {list(STRATEGIES.keys())}')


In [ ]:
from alfworld.agents.environment.alfred_tw_env import AlfredTWEnv

config = {
    'dataset': {
        'data_path': os.path.expanduser('~/.cache/alfworld/json_2.1.1/train'),
        'eval_id_data_path': os.path.expanduser('~/.cache/alfworld/json_2.1.1/valid_seen'),
        'eval_ood_data_path': os.path.expanduser('~/.cache/alfworld/json_2.1.1/valid_unseen'),
        'num_train_games': 0, 'num_eval_games': 0,
    },
    'env': {'goal_desc_human_anns_prob': 0.0, 'task_types': [1,2,3,4,5,6],
            'domain_randomization': False, 'expert_type': 'handcoded'},
    'general': {'training_method': 'dqn', 'random_seed': 42},
    'rl': {'training': {'max_nb_steps_per_episode': MAX_STEPS}},
    'dagger': {'training': {'max_nb_steps_per_episode': MAX_STEPS}},
    'logic': {'domain': os.path.expanduser('~/.cache/alfworld/logic/alfred.pddl'),
              'grammar': os.path.expanduser('~/.cache/alfworld/logic/alfred.twl2')},
}
env_wrapper = AlfredTWEnv(config, train_eval='eval_out_of_distribution')
env = env_wrapper.init_env(batch_size=1)
print(f'ALFWorld: {len(env_wrapper.game_files)} games')


## 2. Pipeline Functions

Each step returns diagnostic information for inspection.


In [ ]:
def retrieve(observation, task, verbose=False):
    """Step 1: Retrieve similar episodes with continuations."""
    query_text = f'{task} | {observation}'
    query_vec = embedder.encode(query_text).tolist()
    results = index.causal_search(vector=query_vec, k=TOP_K, temporal_context=5)

    if verbose:
        print(f'  RETRIEVE: {len(results)} matches')
        for i, r in enumerate(results):
            ep_id = r['entity_id']
            expert_task = task_lookup.get(ep_id, '?')
            n_succ = len(r.get('successors', []))
            print(f'    [{i}] ep={ep_id} score={r["score"]:.3f} task="{expert_task[:40]}" succ={n_succ}')

    return results

def get_full_episode_actions(ep_id):
    """Get the COMPLETE action sequence of an episode from step 0."""
    actions = []
    for step_idx in range(100):
        ts = ep_id * 10000 + step_idx
        a = action_lookup.get(ts, '')
        if a:
            actions.append(a)
        elif step_idx > 0:
            break
    return actions

def format_pipeline_context(results, task_type, candidate_actions):
    """Format context: strategy + FULL expert episode from step 0.

    Key fix: show the complete expert plan from START, not from the
    matched step. The agent needs the full sequence to know what to
    do first, not what comes after a mid-episode match.
    """
    strategy = STRATEGIES.get(task_type, '')
    parts = []
    if strategy:
        parts.append(f'Strategy: {strategy}')

    seen_episodes = set()
    parts.append('Complete expert plans:')
    for r in results[:TOP_K]:
        ep_id = r['entity_id']
        if ep_id in seen_episodes:
            continue
        seen_episodes.add(ep_id)

        full_actions = get_full_episode_actions(ep_id)
        if full_actions:
            parts.append(f'  [{len(seen_episodes)}] {" -> ".join(full_actions[:10])}')

    return '\n'.join(parts) + '\n'

def extract_candidate_actions(results, verbose=False):
    """Step 2: Extract candidate actions from retrieved continuations."""
    actions = []
    for r in results:
        for nid, ts, vec in r.get('successors', [])[:3]:
            action = action_lookup.get(ts, '')
            if action:
                action_type = classify_action(action)
                actions.append({
                    'action': action, 'action_type': action_type,
                    'source_episode': r['entity_id'], 'score': r['score'],
                })
    if verbose:
        print(f'  EXTRACT: {len(actions)} candidate actions')
        type_counts = Counter(a['action_type'] for a in actions)
        print(f'    Types: {dict(type_counts)}')
    return actions

def llm_decide(observation, admissible, task, history, context, verbose=False):
    """Step 5: LLM chooses action."""
    system = ('You are a household agent. You MUST choose EXACTLY ONE action '
              'from the Available actions list below. Copy the action text '
              'EXACTLY as shown. Do NOT invent actions or modify the text.')
    user = f'Task: {task}\n\n'
    if context:
        user += context + '\n'
    if history:
        user += 'Recent:\n'
        for h in history[-5:]:
            user += f'  > {h["action"]} -> {h["obs"][:60]}\n'
        user += '\n'
    user += f'Obs: {observation}\nActions:\n'
    for a in admissible:
        user += f'  - {a}\n'
    user += 'Choose:'

    if verbose:
        print(f'  PROMPT to LLM:')
        print(f'    system: {system[:80]}...')
        print(f'    user ({len(user)} chars):')
        for line in user.split(chr(10))[:15]:
            print(f'      {line[:90]}')
        if user.count(chr(10)) > 15:
            print(f'      ... ({user.count(chr(10))-15} more lines)')

    try:
        resp = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{'role':'system','content':system},{'role':'user','content':user}],
            temperature=0.0, max_tokens=100,
        )
        chosen = resp.choices[0].message.content.strip().lower()
    except:
        chosen = admissible[0].lower()

    for a in admissible:
        if a.lower() == chosen:
            if verbose: print(f'  LLM: exact match "{a}"')
            return a
    for a in admissible:
        if a.lower() in chosen or chosen in a.lower():
            if verbose: print(f'  LLM: partial match "{a}" (raw: "{chosen[:40]}")')
            return a
    if verbose: print(f'  LLM: fallback (raw: "{chosen[:40]}")')
    if verbose: print()
    return admissible[0]

print('Pipeline functions ready (full episode context)')


## 2. Inspection Mode

Run ONE game step-by-step. See every decision the pipeline makes.


In [ ]:
def run_game_inspect(env):
    """Run one game with full diagnostic output."""
    obs, info = env.reset()
    observation = obs[0]
    task = observation.split('Your task is to:')[-1].strip().split('\n')[0] if 'Your task is to:' in observation else ''
    task_type = detect_task_type(task)
    
    print(f'═══ GAME: {task} ═══')
    print(f'Task type: {task_type}')
    print(f'Strategy: {STRATEGIES.get(task_type, "?")}')
    print()
    
    history = []
    step_log = []
    
    for step in range(MAX_STEPS):
        admissible = info['admissible_commands'][0]
        phase = detect_phase(observation, history)
        
        print(f'── Step {step+1} ── phase={phase}')
        print(f'  OBS: {observation[:100]}')
        print(f'  ADMISSIBLE: {admissible[:5]}{"..." if len(admissible)>5 else ""}')
        
        # 1. RETRIEVE
        results = retrieve(observation, task, verbose=True)
        
        # 2. EXTRACT candidate actions
        candidates = extract_candidate_actions(results, verbose=True)
        
        # 3. FORMAT context
        context = format_pipeline_context(results, task_type, candidates)
        
        # 4. LLM DECIDE
        action = llm_decide(observation, admissible, task, history, context, verbose=True)
        
        # Step metadata
        step_info = {
            'step': step, 'phase': phase, 'action': action,
            'action_type': classify_action(action),
            'n_candidates': len(candidates),
            'n_retrievals': len(results),
            'obs': observation[:100],
        }
        step_log.append(step_info)
        
        # Execute
        obs, rewards, dones, info = env.step([action])
        observation = obs[0]
        history.append({'action': action, 'obs': observation[:100]})
        
        print(f'  RESULT: {observation[:80]}')
        print()
        
        if dones[0]:
            break
    
    won = info['won'][0]
    print(f'{"🏆 WIN" if won else "❌ FAIL"} in {len(history)} steps')
    
    return {
        'task': task, 'task_type': task_type, 'won': won,
        'steps': len(history), 'step_log': step_log,
        'history': history,
    }

# Run inspection
result = run_game_inspect(env)


## 3. Evaluation Mode

Run N games, collect aggregate metrics.


In [ ]:
def run_game_eval(env):
    """Run one game silently, return result with metadata."""
    obs, info = env.reset()
    observation = obs[0]
    task = observation.split('Your task is to:')[-1].strip().split('\n')[0] if 'Your task is to:' in observation else ''
    task_type = detect_task_type(task)
    history = []
    phases_seen = []
    action_types_used = []
    
    for step in range(MAX_STEPS):
        admissible = info['admissible_commands'][0]
        phase = detect_phase(observation, history)
        phases_seen.append(phase)
        
        results = retrieve(observation, task)
        candidates = extract_candidate_actions(results)
        context = format_pipeline_context(results, task_type, candidates)
        action = llm_decide(observation, admissible, task, history, context)
        
        action_types_used.append(classify_action(action))
        obs, rewards, dones, info = env.step([action])
        observation = obs[0]
        history.append({'action': action, 'obs': observation[:100]})
        if dones[0]: break
    
    won = info['won'][0]
    return {
        'task': task, 'task_type': task_type, 'won': won,
        'steps': len(history), 'history': history,
        'phases': phases_seen, 'action_types': action_types_used,
    }

# Run evaluation
print(f'Running {N_EVAL} games...')
all_results = []
wins = 0
for g in range(N_EVAL):
    r = run_game_eval(env)
    all_results.append(r)
    if r['won']: wins += 1
    print(f'  {g+1}/{N_EVAL}: {"WIN" if r["won"] else "fail"} ({r["steps"]}s) {r["task_type"]} [{wins}/{g+1}={wins/(g+1)*100:.0f}%]')

print(f'\nResult: {wins}/{N_EVAL} = {wins/N_EVAL*100:.1f}%')


In [ ]:
# Per task type breakdown
print('\n=== Per Task Type ===')
by_type = defaultdict(list)
for r in all_results:
    by_type[r['task_type']].append(r['won'])

for tt in sorted(by_type):
    outcomes = by_type[tt]
    w = sum(outcomes)
    print(f'  {tt}: {w}/{len(outcomes)} = {w/len(outcomes)*100:.0f}%')

# Per phase distribution
print('\n=== Phase Distribution ===')
phase_counts = Counter()
for r in all_results:
    phase_counts.update(r['phases'])
total_phases = sum(phase_counts.values())
for phase, count in phase_counts.most_common():
    print(f'  {phase}: {count} ({count/total_phases*100:.0f}%)')

# Action type distribution in wins vs fails
print('\n=== Action Types: Wins vs Fails ===')
win_actions = Counter()
fail_actions = Counter()
for r in all_results:
    c = win_actions if r['won'] else fail_actions
    c.update(r['action_types'])
print('  Wins:', dict(win_actions.most_common(5)))
print('  Fails:', dict(fail_actions.most_common(5)))


## 4. Error Analysis

Post-hoc study of failures: why did the agent fail?


In [ ]:
# === Error Analysis ===
failures = [r for r in all_results if not r['won']]
print(f'{len(failures)} failures to analyze')

# Failure patterns
print('\n=== Failure Patterns ===')

# 1. How many steps before failing?
step_counts = [r['steps'] for r in failures]
print(f'  Mean steps before failure: {np.mean(step_counts):.1f} (max={MAX_STEPS})')
print(f'  Timeout failures (hit MAX_STEPS): {sum(1 for s in step_counts if s >= MAX_STEPS)}')

# 2. What phase were they in when they failed?
print('\n=== Final Phase at Failure ===')
final_phases = Counter(r['phases'][-1] if r['phases'] else '?' for r in failures)
for phase, count in final_phases.most_common():
    print(f'  {phase}: {count}')

# 3. What task types fail most?
print('\n=== Failure Rate by Task Type ===')
for tt in sorted(by_type):
    outcomes = by_type[tt]
    fail_rate = 1 - sum(outcomes)/len(outcomes)
    print(f'  {tt}: {fail_rate*100:.0f}% failure rate ({len(outcomes)} games)')

# 4. Repetitive actions (agent stuck in loop)
print('\n=== Stuck in Loop? (same action 3+ times in a row) ===')
loop_count = 0
for r in failures:
    actions = [h['action'] for h in r['history']]
    for i in range(len(actions)-2):
        if actions[i] == actions[i+1] == actions[i+2]:
            loop_count += 1
            print(f'  Game "{r["task"][:30]}": repeated "{actions[i]}" at step {i+1}')
            break
print(f'  Total games with loops: {loop_count}/{len(failures)}')

# 5. Did retrieval help at all? Compare steps where candidates existed
print('\n=== Retrieval Quality ===')
# TODO: add retrieval metadata to step_log for deeper analysis
print('  (Enable inspection mode for per-step retrieval analysis)')


In [ ]:
# === Inspect worst failures ===
print('=== Worst Failures (full action history) ===')
# Sort failures by step count (shortest = worst, couldn\'t even start)
worst = sorted(failures, key=lambda r: r['steps'])[:3]

for r in worst:
    print(f'\nTask: {r["task"]}')
    print(f'Type: {r["task_type"]}, Steps: {r["steps"]}')
    for i, h in enumerate(r['history'][:10]):
        print(f'  {i+1}. {h["action"]} → {h["obs"][:60]}')
    if len(r['history']) > 10:
        print(f'  ... ({len(r["history"])-10} more steps)')
    print()


In [ ]:
# Save for comparison across runs
results_data = {
    'config': {'model': LLM_MODEL, 'n_eval': N_EVAL, 'top_k': TOP_K},
    'summary': {'wins': wins, 'total': N_EVAL, 'rate': wins/N_EVAL},
    'per_task': {tt: {'wins': sum(o), 'total': len(o)} for tt, o in by_type.items()},
    'games': [{'task': r['task'], 'task_type': r['task_type'], 'won': r['won'], 
               'steps': r['steps']} for r in all_results],
}

with open('../data/episodic/e9_pipeline_results.json', 'w') as f:
    json.dump(results_data, f, indent=2)
print(f'Saved to data/episodic/e9_pipeline_results.json')
